In [1]:
from sentence_transformers import SentenceTransformer
import numpy as np
from scipy.special import softmax

/home/su_dalle-lucca-tosi/.pyenv/versions/3.11.0/envs/baf/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Example data
X = ["Will Smith", "Martin Lawrence", "Francis Coppola"]
Z_baseline = ["Question about Will Smith", "Movie with Martin Lawrence", "Director Francis Coppola"]
Z_masked = ["Question about AD_HOC", "Movie with NODE_VALUE", "Director NODE_VALUE"]

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def mutual_information(x_list, z_list):
    X_emb = model.encode(x_list)
    Z_emb = model.encode(z_list)
    sims = np.exp(np.dot(X_emb, Z_emb.T) / (np.linalg.norm(X_emb, axis=1)[:,None] * np.linalg.norm(Z_emb, axis=1)))
    p_xz = sims / sims.sum()
    p_x = p_xz.sum(axis=1, keepdims=True)
    p_z = p_xz.sum(axis=0, keepdims=True)
    I = (p_xz * np.log(p_xz / (p_x @ p_z))).sum()
    return I

I_baseline = mutual_information(X, Z_baseline)
I_masked = mutual_information(X, Z_masked)
privacy_gain = (I_baseline - I_masked) / I_baseline * 100

print(f"I_baseline={I_baseline:.4f}, I_masked={I_masked:.4f}, Privacy gain={privacy_gain:.2f}%")

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 18b38d53-7e87-4013-a14f-ceee3d7a4f38)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


I_baseline=0.0472, I_masked=0.0004, Privacy gain=99.07%


## MetaQA data

### Values from KG

In [3]:
#Download dataset from: https://drive.google.com/drive/folders/0B-36Uca2AvwhTWVFSUZqRXVtbUE?resourcekey=0-kdv6ho5KcpEXdI2aUdLn_g
dataset_folder = ""
dataset_file = dataset_folder + "METAQA/kb.txt"

In [4]:
with open(dataset_file) as file:
    lines = [line.rstrip().split('|') for line in file]

In [5]:
movies = {}

for l in lines:
    movie_id, relation, person = l  # unpack for readability

    # --- Ensure movie exists ---
    if movie_id not in movies:
        movies[movie_id] = {}
    movie_dict = movies[movie_id]

    # --- Add to movie dictionary ---
    movie_dict.setdefault(relation, []).append(person)


In [6]:
sensitive_values_kg = []
for name, attrs in movies.items():
    sensitive_values_kg.append(name)
    
    for i in list(attrs.values()):
        for j in i:
            sensitive_values_kg.append(j)

In [7]:
len(sensitive_values_kg)

151168

### Values from questions

In [8]:
import pandas as pd

df_baf = pd.read_csv('baf_complete.tsv', sep="\t")
df_baf.columns = ["question", "entities", "mask", "answer_masked", "answer_unmasked"]
sample_questions = df_baf['question']

In [9]:
from typing import List
import re
def extract_bracketed_entities(sentence: str) -> List[str]:
    """
    Extract entities wrapped in square brackets [...] from the sentence.
    Returns a list of unique values (without brackets).
    """
    return list(re.findall(r'\[([^\[\]]+)\]', sentence))

In [10]:
sensitive_values_qa = []
for question in sample_questions:
    sensitives = extract_bracketed_entities(question)
    sensitive_values_qa = sensitive_values_qa + sensitives

In [11]:
len(sensitive_values_qa)

502

## Run

In [12]:
sensitive_values = list(set(sensitive_values_kg + sensitive_values_qa))

In [13]:
len(sensitive_values)

43234

In [14]:
len(sample_questions)

502

In [15]:
df_baf = pd.read_csv('baf_complete.tsv', sep="\t")
df_baf.columns = ["question", "entities", "mask", "answer_masked", "answer_unmasked"]

In [23]:
questions = []
for q in df_baf['question']:
    q = q.replace('[', '').replace(']','')
    questions.append(q)

In [27]:
I_baseline = mutual_information(sensitive_values_qa, questions)

In [28]:
I_masked = mutual_information(sensitive_values_qa, df_baf['mask'])

In [29]:
privacy_gain = (I_baseline - I_masked) / I_baseline * 100
print(f"I_baseline={I_baseline:.4f}, I_masked={I_masked:.4f}, Privacy gain={privacy_gain:.2f}%")

I_baseline=0.0024, I_masked=0.0007, Privacy gain=71.10%
